# Đánh giá V4: LOSO trên MIMIC, kiểm định chéo MIMIC ↔ AFDB, chấm bộ dò nhịp bằng ECG

Ba phép kiểm định làm nên phần "không rò rỉ dữ liệu" của V4, gộp từ ba script cũ `run_v4_benchmark.py`, `run_cross_dataset.py`, `run_beat_validation.py`. Đầu vào là bảng đặc trưng từ `training/00`; đầu ra ghi vào `results/`.

| Phần | Câu hỏi | Đầu ra |
|---|---|---|
| 1. LOSO 35 bệnh nhân | Mô hình nhận ra bệnh nhân **chưa từng thấy** không? | `results/v4/benchmark_v4/` |
| 2. Cross-dataset | Train trên PPG (MIMIC) có đúng trên ECG (AFDB) và ngược lại không? | `results/v4/cross_dataset/` |
| 3. Beat validation | Bộ dò nhịp PPG có khớp R-peak của ECG ghi song song không? | `results/v4/beat_validation/` |

Thời gian: phần 1 và 2 có GridSearch trong từng fold, khoảng 5 đến 15 phút; phần 3 vài phút.

In [1]:
# Tìm gốc repo và nạp thư viện src/healthsense_core (chạy được từ bất kỳ thư mục con nào)
import os, sys, json, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "healthsense_core").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Không tìm thấy gốc repo (thư mục chứa src/healthsense_core)")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

Repo root: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab


In [2]:
from healthsense_core import config, afdb, evaluation
from healthsense_core.feature_extraction import load_features as load_mimic
from healthsense_core.training import run_loso_benchmark, build_model_registry, iqr_train_mask
from healthsense_core.beat_validation import validate_all

FEATURE_COLS = config.CORE_FEATURES      # 13 đặc trưng, đã bỏ nhóm LF
mimic = load_mimic()
print(f"MIMIC: {len(mimic)} cửa sổ, {mimic.record_id.nunique()} bệnh nhân | đặc trưng: {len(FEATURE_COLS)}")

MIMIC: 4130 cửa sổ, 35 bệnh nhân | đặc trưng: 13


## 1. Leave-One-Subject-Out trên MIMIC

Mỗi fold giữ trọn một bệnh nhân làm test; lọc IQR và scaler chỉ fit trên train của fold; GridSearchCV với GroupKFold 3 bên trong để chọn tham số mà không nhìn bệnh nhân test. Metric báo ở hai mức: cửa sổ 30 giây và bệnh nhân (trung bình xác suất các cửa sổ).

In [3]:
t0 = time.time()
predictions = run_loso_benchmark(mimic, feature_cols=FEATURE_COLS)
print(f"LOSO xong trong {(time.time()-t0)/60:.1f} phút")
os.makedirs(config.BENCHMARK_V4_DIR, exist_ok=True)
predictions.to_csv(os.path.join(config.BENCHMARK_V4_DIR, "loso_predictions.csv"), index=False)
summary = evaluation.summarize(predictions)
summary.to_csv(os.path.join(config.BENCHMARK_V4_DIR, "benchmark_results_v4.csv"), index=False)
cols = [c for c in ["Model", "Level", "Accuracy", "Recall (Sensitivity)", "Specificity", "F1-Score", "ROC-AUC", "FN", "FP"] if c in summary.columns]
summary[cols].round(4)


🏋️ Logistic Regression — LOSO 35 folds (nested GroupKFold(3) tuning)...


    fold 10/35 xong...


    fold 20/35 xong...


    fold 30/35 xong...



🏋️ Random Forest — LOSO 35 folds (nested GroupKFold(3) tuning)...


...

    fold 10/35 xong...


    fold 20/35 xong...


    fold 30/35 xong...


LOSO xong trong 2.8 phút


,Model,Level,Accuracy,Recall (Sensitivity),Specificity,F1-Score,ROC-AUC,FN,FP
0,Logistic Regression,window,0.9228,0.9822,0.8522,0.9325,0.8707,40,279
1,Logistic Regression,subject,0.9429,1.0000,0.8750,0.9500,0.8750,0,2
2,Random Forest,window,0.9213,0.9746,0.8581,0.9308,0.9398,57,268
3,Random Forest,subject,0.9429,1.0000,0.8750,0.9500,0.9309,0,2
4,XGBoost,window,0.9182,0.9630,0.8649,0.9274,0.9129,83,255
5,XGBoost,subject,0.9429,1.0000,0.8750,0.9500,0.9013,0,2


In [4]:
for level in ("subject", "window"):
    evaluation.plot_confusion_matrices(predictions, config.BENCHMARK_V4_DIR, level)
    evaluation.plot_roc_curves(predictions, config.BENCHMARK_V4_DIR, level)
evaluation.plot_subject_probabilities(predictions, config.BENCHMARK_V4_DIR)
print("Hình đã lưu vào", os.path.relpath(config.BENCHMARK_V4_DIR))

Hình đã lưu vào results\benchmark_v4


## 2. Kiểm định chéo MIMIC (PPG) ↔ AFDB (ECG)

Bài kiểm tra tổng quát hoá khắc nghiệt nhất của V4: khác bệnh viện, khác loại cảm biến, khác quần thể. Tiền xử lý và tuning chỉ dùng dataset train; dataset test giữ nguyên vẹn.

In [5]:
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix

afdb_df = afdb.load_features()
print(f"AFDB: {len(afdb_df)} cửa sổ, {afdb_df.record_id.nunique()} bản ghi")

def evaluate_direction(train_df, test_df, train_name, test_name):
    X_tr, y_tr, g_tr = train_df[FEATURE_COLS], train_df["status"].to_numpy(), train_df["record_id"].to_numpy()
    X_te, y_te = test_df[FEATURE_COLS], test_df["status"].to_numpy()
    keep = iqr_train_mask(X_tr).to_numpy()
    rows = []
    for name, spec in build_model_registry().items():
        search = GridSearchCV(spec["pipeline"], spec["param_grid"], cv=GroupKFold(config.INNER_CV_FOLDS),
                              scoring="roc_auc", n_jobs=-1, refit=True)
        search.fit(X_tr[keep], y_tr[keep], groups=g_tr[keep])
        probs = search.predict_proba(X_te)[:, 1]; preds = (probs >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, preds, labels=[0, 1]).ravel()
        rows.append({"Direction": f"{train_name} -> {test_name}", "Model": name,
                     "Accuracy": accuracy_score(y_te, preds), "Recall": recall_score(y_te, preds, zero_division=0),
                     "Specificity": tn / (tn + fp) if tn + fp else 0.0, "Precision": precision_score(y_te, preds, zero_division=0),
                     "F1": f1_score(y_te, preds, zero_division=0), "ROC-AUC": roc_auc_score(y_te, probs),
                     "FN": int(fn), "FP": int(fp), "Best Params": str(search.best_params_)})
        print(f"  {train_name} -> {test_name}  {name:<20} AUROC {rows[-1]['ROC-AUC']:.4f}")
    return rows

t0 = time.time()
cross = pd.DataFrame(evaluate_direction(mimic, afdb_df, "MIMIC", "AFDB") + evaluate_direction(afdb_df, mimic, "AFDB", "MIMIC"))
out_dir = os.path.join(config.RESULTS_DIR, "v4", "cross_dataset"); os.makedirs(out_dir, exist_ok=True)
cross.to_csv(os.path.join(out_dir, "cross_dataset_results.csv"), index=False)
print(f"Xong trong {(time.time()-t0)/60:.1f} phút")
cross.drop(columns=["Best Params"]).round(4)

AFDB: 28734 cửa sổ, 25 bản ghi


  MIMIC -> AFDB  Logistic Regression  AUROC 0.9757


  MIMIC -> AFDB  Random Forest        AUROC 0.9867


  MIMIC -> AFDB  XGBoost              AUROC 0.9904


  AFDB -> MIMIC  Logistic Regression  AUROC 0.8807


  AFDB -> MIMIC  Random Forest        AUROC 0.9788


  AFDB -> MIMIC  XGBoost              AUROC 0.9715
Xong trong 0.3 phút


,Direction,Model,Accuracy,Recall,Specificity,Precision,F1,ROC-AUC,FN,FP
0,MIMIC -> AFDB,Logistic Regression,0.9437,0.9793,0.9215,0.8858,0.9302,0.9757,228,1391
1,MIMIC -> AFDB,Random Forest,0.9435,0.9702,0.9269,0.8920,0.9295,0.9867,328,1295
2,MIMIC -> AFDB,XGBoost,0.9508,0.9836,0.9303,0.8978,0.9387,0.9904,181,1234
3,AFDB -> MIMIC,Logistic Regression,0.9104,0.9719,0.8374,0.8765,0.9217,0.8807,63,307
4,AFDB -> MIMIC,Random Forest,0.9334,0.9719,0.8877,0.9113,0.9406,0.9788,63,212
5,AFDB -> MIMIC,XGBoost,0.9213,0.9737,0.8591,0.8914,0.9307,0.9715,59,266


## 3. Chấm bộ dò nhịp PPG bằng R-peak trên ECG

MIMIC PERform ghi ECG song song với PPG, nên có đáp án chuẩn cho từng nhịp. Dung sai khớp nhịp ±150 ms sau khi bù trễ truyền mạch. Phần này cũng tính RMSSD và pNN50 từ chính ECG để thẩm định hai bệnh nhân "Normal" bị nghi gán nhãn sai.

In [6]:
t0 = time.time()
beats = validate_all(verbose=False)
out_dir = os.path.join(config.RESULTS_DIR, "v4", "beat_validation"); os.makedirs(out_dir, exist_ok=True)
beats.to_csv(os.path.join(out_dir, "beat_validation.csv"), index=False)
print(f"Xong trong {(time.time()-t0)/60:.1f} phút")

ok = beats[~beats["ecg_unreliable"]]
print(f"{len(ok)}/{len(beats)} bệnh nhân có ECG đủ tin cậy để chấm")
display(ok.groupby("label")[["f1", "sensitivity", "ppv", "hr_mae_bpm"]].median().round(3))
print("5 bệnh nhân dò nhịp kém nhất:")
display(ok.nsmallest(5, "f1")[["record_id", "label", "f1", "sensitivity", "ppv", "hr_mae_bpm"]].round(3))

suspects = ["mimic_perform_non_af_012", "mimic_perform_non_af_014"]
normals = beats[(beats.label == "Normal") & (~beats.record_id.isin(suspects))]
afs = beats[beats.label == "AFib"]
print(f"Normal khác (median): RMSSD={normals.ecg_RMSSD.median():.0f} ms, pNN50={normals.ecg_pNN50.median():.0f}%")
print(f"AFib (median)       : RMSSD={afs.ecg_RMSSD.median():.0f} ms, pNN50={afs.ecg_pNN50.median():.0f}%")
display(beats[beats.record_id.isin(suspects)][["record_id", "label", "ecg_RMSSD", "ecg_pNN50", "ecg_unreliable"]].round(0))

Xong trong 0.1 phút
35/35 bệnh nhân có ECG đủ tin cậy để chấm


,f1,sensitivity,ppv,hr_mae_bpm
label,,,,
AFib,0.855,0.833,0.885,4.7
Normal,0.991,0.990,0.991,1.1


5 bệnh nhân dò nhịp kém nhất:


,record_id,label,f1,sensitivity,ppv,hr_mae_bpm
7,mimic_perform_af_008,AFib,0.107,0.107,0.106,4.70
32,mimic_perform_non_af_014,Normal,0.196,0.226,0.173,23.95
30,mimic_perform_non_af_012,Normal,0.355,0.294,0.449,29.90
13,mimic_perform_af_014,AFib,0.440,0.439,0.440,1.25
12,mimic_perform_af_013,AFib,0.482,0.479,0.485,1.55


Normal khác (median): RMSSD=47 ms, pNN50=6%
AFib (median)       : RMSSD=170 ms, pNN50=74%


,record_id,label,ecg_RMSSD,ecg_pNN50,ecg_unreliable
30,mimic_perform_non_af_012,Normal,233.0,41.0,False
32,mimic_perform_non_af_014,Normal,31.0,2.0,False
